# `mutate` — Reference
`mutate` creates or overwrites columns using clean keyword argument syntax (`col="expr"` or `col=callable`). Each entry is evaluated in order via pandas `eval()` — a plain formula per column, or a lambda when needed.
### Calling Styles
1. **Keyword arguments** (most Pythonic): `.pt.mutate(bmi="body_mass_g / bill_length_mm ** 2", mass_kg="body_mass_g / 1000")`
2. **Dictionary unpacking** (for new column names with spaces): `.pt.mutate(**{"body mass kg": "body_mass_g / 1000"})`
3. **External file specification**: `.pt.mutate("@features.txt")`
---


In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import numpy as np
import pytae as pt

penguins = pt.sample_data['penguins']

## A single derived column

In [2]:
# Body mass index style ratio — clean assignment formula via kwargs
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.select('species', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,body_mass_g,bill_length_mm,bmi
77,Adelie,3900.0,37.2,2.818245
320,Gentoo,4850.0,48.5,2.061856
289,Gentoo,5550.0,50.7,2.159121
111,Adelie,4600.0,45.6,2.212219
253,Gentoo,6050.0,59.6,1.703189
128,Adelie,3050.0,39.0,2.005260
88,Adelie,3950.0,38.3,2.692772
259,Gentoo,5350.0,48.7,2.255775
49,Adelie,4150.0,42.3,2.319356
23,Adelie,3950.0,38.2,2.706889


In [3]:
# Column names inside the expression must stay unquoted — quoting one turns it into
# a string literal, which will break arithmetic
try:
    penguins.pt.mutate(bmi="'body_mass_g' / 'bill_length_mm' ** 2")
except TypeError as exc:
    print(f"TypeError (as expected): {exc}")


TypeError (as expected): unsupported operand type(s) for ** or pow(): 'str' and 'int'


## Multiple entries in one call, and chaining a later entry off an earlier one
Entries are applied left to right, so a later expression can reference a column derived earlier in the *same* `mutate()` call.

In [4]:
# Two independent derived columns in one call via kwargs
(
    penguins
    .pt.mutate(heavy='body_mass_g > 4000', mass_kg='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'heavy')
    .sample(10)
)


,species,body_mass_g,mass_kg,heavy
61,Adelie,4400.0,4.400,True
175,Chinstrap,3800.0,3.800,False
24,Adelie,3800.0,3.800,False
54,Adelie,2900.0,2.900,False
342,Gentoo,5200.0,5.200,True
270,Gentoo,4850.0,4.850,True
2,Adelie,3250.0,3.250,False
284,Gentoo,4700.0,4.700,True
123,Adelie,3875.0,3.875,False
279,Gentoo,5550.0,5.550,True


In [5]:
# mass_lb references mass_kg, derived by the keyword just before it
(
    penguins
    .pt.mutate(mass_kg='body_mass_g / 1000', mass_lb='mass_kg * 2.20462')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'mass_lb')
    .sample(10)
)


,species,body_mass_g,mass_kg,mass_lb
220,Gentoo,4500.0,4.500,9.920790
238,Gentoo,4800.0,4.800,10.582176
202,Chinstrap,3325.0,3.325,7.330361
100,Adelie,3725.0,3.725,8.212210
330,Gentoo,5000.0,5.000,11.023100
65,Adelie,3950.0,3.950,8.708249
231,Gentoo,5550.0,5.550,12.235641
165,Chinstrap,4050.0,4.050,8.928711
243,Gentoo,5050.0,5.050,11.133331
8,Adelie,3475.0,3.475,7.661054


## String comparisons and local variables
String literals *inside* the expression (e.g. `'Adelie'`) need real quotes — column names stay bare. A variable from the calling scope can be referenced with an `@` prefix, same as pandas' own `eval()`/`query()`.


In [6]:
# String comparisons inside the expression formula
(
    penguins
    .pt.mutate(is_adelie="species == 'Adelie'")
    .pt.select('species', 'is_adelie')
    .sample(10)
)


,species,is_adelie
163,Chinstrap,False
154,Chinstrap,False
141,Adelie,True
124,Adelie,True
20,Adelie,True
328,Gentoo,False
226,Gentoo,False
89,Adelie,True
203,Chinstrap,False
76,Adelie,True


In [7]:
# @-prefixed names resolve against the scope that called mutate(), not the module
# — works identically whether calling pt.mutate() or chaining .pt.mutate()
threshold = 4000

(
    penguins
    .pt.mutate(heavy='body_mass_g >= @threshold')
    .pt.select('species', 'body_mass_g', 'heavy')
    .sample(10)
)


,species,body_mass_g,heavy
198,Chinstrap,3400.0,False
206,Chinstrap,3350.0,False
30,Adelie,3250.0,False
21,Adelie,3600.0,False
124,Adelie,3050.0,False
207,Chinstrap,3450.0,False
57,Adelie,3800.0,False
245,Gentoo,5100.0,True
116,Adelie,2900.0,False
181,Chinstrap,4550.0,True


## Overwriting an existing column

In [8]:
# mutate() can overwrite a column in place, e.g. converting units
(
    penguins
    .pt.mutate(body_mass_g='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g')
    .sample(10)
)


,species,body_mass_g
234,Gentoo,4.200
130,Adelie,3.325
296,Gentoo,4.600
71,Adelie,3.900
252,Gentoo,5.050
141,Adelie,3.475
148,Adelie,3.450
183,Chinstrap,4.300
140,Adelie,3.400
142,Adelie,3.050


## Column names with spaces — backtick quoting
`pandas.eval()` uses **backticks**, not the single/double quotes used elsewhere in pytae, to reference a column name containing a space.

In [9]:
import pandas as pd
spaced = pd.DataFrame({'body mass g': [3750, 4200], 'bill length mm': [39.1, 46.5]})

# Backticks protect column names that contain spaces
spaced.pt.mutate(bmi="`body mass g` / `bill length mm` ** 2")


,body mass g,bill length mm,bmi
0,3750,39.1,2.452888
1,4200,46.5,1.942421


## Real-world pipeline: mutate → filter → select
`mutate()` chains like any other pytae method — filter on a column you just derived with `qry()`.

In [10]:
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.qry(bmi='> 2')
    .pt.select('species', 'island', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,island,body_mass_g,bill_length_mm,bmi
125,Adelie,Torgersen,4000.0,40.6,2.426654
148,Adelie,Dream,3450.0,36.0,2.662037
151,Adelie,Dream,4000.0,41.5,2.322543
28,Adelie,Biscoe,3150.0,37.9,2.192967
90,Adelie,Dream,3550.0,35.7,2.785428
245,Gentoo,Biscoe,5100.0,46.1,2.399763
286,Gentoo,Biscoe,4650.0,46.2,2.178557
85,Adelie,Dream,3550.0,41.3,2.081269
262,Gentoo,Biscoe,4300.0,45.3,2.095425
41,Adelie,Dream,3900.0,40.8,2.342849


## Conditional column creation — `if_else()`, `case_when()`, `map()`
Plain `eval()` has no ternary/`where()` support, so `mutate()` provides three dplyr-style helpers as ordinary function calls, evaluated via `np.where()`/`np.select()`/`Series.map()`: `if_else(condition, true_value, false_value)`, `case_when((cond1, val1), (cond2, val2), ..., default)`, and `map(column, {key: value, ...}, default)`. A trailing bare argument to `case_when` is the catch-all default (like SQL ELSE). Because they are ordinary calls, they compose and chain with each other and with any pandas method. String outcomes need quotes; conditions are vectorized — prefer `and`/`or`/`not` (the bitwise `&`/`|`/`~` also work).

In [11]:
# if_else(condition, true_value, false_value) — like dplyr's if_else()
# true_value and false_value can be scalars or column names; conditions are vectorized
(
    penguins
    .pt.mutate(weight_class="if_else(body_mass_g > 4000, 'heavy', 'light')")
    .pt.select('species', 'body_mass_g', 'weight_class')
    .sample(10)
)


,species,body_mass_g,weight_class
79,Adelie,4000.0,light
129,Adelie,4000.0,light
290,Gentoo,4750.0,heavy
284,Gentoo,4700.0,heavy
263,Gentoo,4750.0,heavy
181,Chinstrap,4550.0,heavy
264,Gentoo,5550.0,heavy
328,Gentoo,4575.0,heavy
126,Adelie,3275.0,light
33,Adelie,3900.0,light


In [12]:
# case_when supports tuples or flat pairs with default=
(
    penguins
    .pt.mutate(
        size_class="case_when(body_mass_g >= 4500, 'large', body_mass_g >= 3500, 'medium', default='small')"
    )
    .pt.select('species', 'body_mass_g', 'size_class')
    .sample(10)
)


,species,body_mass_g,size_class
315,Gentoo,5200.0,large
127,Adelie,4300.0,medium
190,Chinstrap,2700.0,small
75,Adelie,4250.0,medium
280,Gentoo,4200.0,medium
333,Gentoo,5500.0,large
183,Chinstrap,4300.0,medium
56,Adelie,3550.0,medium
133,Adelie,4475.0,medium
296,Gentoo,4600.0,large


In [13]:
# map(column, {key: value, ...}, default) — recode a column through a dictionary lookup
# unmapped keys become default if given, else NaN
(
    penguins
    .pt.mutate(
        island_code="map(island, {'Torgersen': 'TOR', 'Biscoe': 'BIS'}, 'OTH')"
    )
    .pt.select('species', 'island', 'island_code')
    .sample(10)
)


,species,island,island_code
336,Gentoo,Biscoe,BIS
84,Adelie,Dream,OTH
75,Adelie,Torgersen,TOR
9,Adelie,Torgersen,TOR
296,Gentoo,Biscoe,BIS
222,Gentoo,Biscoe,BIS
301,Gentoo,Biscoe,BIS
42,Adelie,Dream,OTH
72,Adelie,Torgersen,TOR
308,Gentoo,Biscoe,BIS


## First non-null resolution — `coalesce()`

`coalesce(col1, col2, ..., default)` evaluates candidates left-to-right and returns the first non-null value per row, like SQL `COALESCE()` or `dplyr::coalesce()`:

In [14]:
contacts = pd.DataFrame({
    'mobile': [None, '555-1234', None],
    'home': ['555-5678', None, None],
    'work': [None, None, '555-9012'],
})

contacts.pt.mutate(preferred_contact="coalesce(mobile, home, work, 'N/A')")


,mobile,home,work,preferred_contact
0,NaN,555-5678,NaN,555-5678
1,555-1234,NaN,NaN,555-1234
2,NaN,NaN,555-9012,555-9012


## Combining expressions and callables in kwargs

`mutate()` cleanly mixes expression strings and python callables (e.g. lambdas) in keyword arguments:


In [15]:
(
    penguins
    .pt.mutate(
        bmi='body_mass_g / bill_length_mm ** 2',
        mass_kg='body_mass_g / 1000',
        is_heavy=lambda df: df['body_mass_g'] > 4000,
    )
    .pt.select('species', 'bmi', 'mass_kg', 'is_heavy')
    .head(5)
)


,species,bmi,mass_kg,is_heavy
0,Adelie,2.452888,3.75,False
1,Adelie,2.435507,3.80,False
2,Adelie,2.001121,3.25,False
3,Adelie,NaN,NaN,False
4,Adelie,2.561456,3.45,False


## Loading specs from an external file — `@specs.txt`
For complex feature engineering or shared pipelines across Python and the CLI, specs can be loaded from an external file. The file supports multiline expressions, `#` comments, empty lines, and column chaining (where later formulas reference columns defined earlier in the same file).
### File content of `penguin_features.txt`:
```text
# Engineering features for penguins
mass_kg = body_mass_g / 1000
# Convert kg to lbs (references mass_kg computed above)
mass_lb = mass_kg * 2.20462
```


In [16]:
# Write a small demo spec file with comments and assignment '=' syntax
spec_content = """# Engineering features for penguins
mass_kg = body_mass_g / 1000

# Convert kg to lbs
mass_lb = mass_kg * 2.20462
"""
with open("penguin_features.txt", "w") as f:
    f.write(spec_content)

# Load directly using @filename
result = penguins.pt.mutate("@penguin_features.txt")
out = (
    result
    .pt.select("species", "mass_kg", "mass_lb")
    .sample(5)
)

# Clean up demo file
import os
os.remove("penguin_features.txt")

out


,species,mass_kg,mass_lb
293,Gentoo,5.20,11.464024
62,Adelie,3.60,7.936632
216,Chinstrap,3.40,7.495708
173,Chinstrap,3.40,7.495708
118,Adelie,3.35,7.385477


## Columns with spaces — SQL-style brackets `[col]`

SQL-style square brackets `[col a]` are supported alongside backticks, avoiding shell backtick substitution hazards:

In [17]:
spaced.pt.mutate(ratio='[body mass g] / [bill length mm]')


,body mass g,bill length mm,ratio
0,3750,39.1,95.907928
1,4200,46.5,90.322581


## Creating new columns with spaces — string expressions `[col] = ...`

You can create columns with spaces directly using string assignment expressions with brackets `[new col] = ...` with zero dictionary unpacking.

(Dictionary unpacking `**{'col name': ...}` is also supported for kwargs).

In [18]:
# Create a new column with spaces using bracket string expression (no dict unpacking!)
(
    spaced
    .pt.mutate("[body mass ratio] = [body mass g] / [bill length mm]")
    .pt.select('[body mass g]', '[body mass ratio]')
    .head()
)

,body mass g,body mass ratio
0,3750,95.907928
1,4200,90.322581


In [19]:
# Define multiple new columns with spaces at once via string expressions
(
    penguins
    .pt.mutate(
        "[body mass kg] = body_mass_g / 1000",
        "[bill ratio] = bill_length_mm / bill_depth_mm"
    )
    .pt.select('species', '[body mass kg]', '[bill ratio]')
    .head()
)

,species,body mass kg,bill ratio
0,Adelie,3.75,2.090909
1,Adelie,3.80,2.270115
2,Adelie,3.25,2.238889
3,Adelie,NaN,NaN
4,Adelie,3.45,1.901554


Note: When loading specs from an external file (`@specs.txt`) or via the CLI (`-mutate`), names with spaces can be written directly on the left-hand side of `=` without dictionary unpacking:

```text
# inside an external spec file or CLI flag:
my bmi = [body mass g] / [bill length mm] ** 2
```
